# CS 521 HW4, Problem 3: PPO and LLM Post-Training

In [ ]:
from importlib.util import find_spec
import subprocess
import sys

missing = [package for module, package in
           [("gymnasium", "gymnasium"), ("transformers", "transformers==4.52.4")]
           if find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## (a) PPO: Setup

In [ ]:
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

torch.set_num_threads(1)

CONFIG = dict(
    total_steps=200_000,
    num_envs=4,
    num_steps=128,
    lr=2.5e-4,
    anneal_lr=True,
    gamma=0.99,
    gae_lambda=0.95,
    num_minibatches=4,
    update_epochs=4,
    clip_coef=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    norm_adv=True,
)
SEEDS = [0, 1, 2, 3, 4]

### Environments

In [ ]:
class Envs:
    def __init__(self, n, seed):
        self.envs = [gym.make("CartPole-v1") for _ in range(n)]
        self.obs = np.stack([e.reset(seed=seed + i)[0] for i, e in enumerate(self.envs)])
        for i, e in enumerate(self.envs):
            e.action_space.seed(seed + i)
        self.ep_return = np.zeros(n)
        self.global_step = 0
        self.episodes = []

    def step(self, actions):
        rewards, terms, truncs, final_obs = [], [], [], []
        for i, (e, a) in enumerate(zip(self.envs, actions)):
            o, r, term, trunc, _ = e.step(int(a))
            self.ep_return[i] += r
            fo = None
            if term or trunc:
                fo = o
                self.episodes.append((self.global_step + i + 1, self.ep_return[i]))
                self.ep_return[i] = 0
                o, _ = e.reset()
            self.obs[i] = o
            rewards.append(r); terms.append(term); truncs.append(trunc); final_obs.append(fo)
        self.global_step += len(self.envs)
        return self.obs.copy(), np.array(rewards), np.array(terms), np.array(truncs), final_obs

### Actor and critic

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias=0.0):
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias)
    return layer


class Agent(nn.Module):
    def __init__(self, obs_dim=4, n_actions=2):
        super().__init__()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0))
        self.actor = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, n_actions), std=0.01))

    def get_value(self, x):
        return self.critic(x).squeeze(-1)

    def get_action_and_value(self, x, action=None):
        dist = Categorical(logits=self.actor(x))
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), self.get_value(x)

### PPO training

In [ ]:
def compute_gae(rewards, values, dones, next_value, next_done, gamma, lam):


    T = len(rewards)
    adv = torch.zeros_like(rewards)
    last = 0
    for t in reversed(range(T)):
        not_done = 1.0 - (next_done if t == T - 1 else dones[t + 1])
        nv = next_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * nv * not_done - values[t]
        last = delta + gamma * lam * not_done * last
        adv[t] = last
    return adv, adv + values


def ppo_update(agent, opt, b, cfg):
    n = len(b["obs"])
    mb_size = n // cfg["num_minibatches"]
    stats = []
    for _ in range(cfg["update_epochs"]):
        idx = torch.randperm(n)
        for start in range(0, n, mb_size):
            mb = idx[start:start + mb_size]
            _, logp, ent, v = agent.get_action_and_value(b["obs"][mb], b["actions"][mb])
            ratio = (logp - b["logp"][mb]).exp()
            adv = b["adv"][mb]
            if cfg["norm_adv"]:
                adv = (adv - adv.mean()) / (adv.std() + 1e-8)
            pg_loss = torch.max(-adv * ratio,
                                -adv * ratio.clamp(1 - cfg["clip_coef"], 1 + cfg["clip_coef"])).mean()
            v_loss = 0.5 * ((v - b["returns"][mb]) ** 2).mean()
            loss = pg_loss - cfg["ent_coef"] * ent.mean() + cfg["vf_coef"] * v_loss
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), cfg["max_grad_norm"])
            opt.step()
            stats.append(((ratio - 1).abs() > cfg["clip_coef"]).float().mean().item())
    return {"clip_frac": float(np.mean(stats))}


def train_ppo(seed, cfg=None, verbose=False):
    cfg = {**CONFIG, **(cfg or {})}
    torch.manual_seed(seed)
    np.random.seed(seed)
    envs = Envs(cfg["num_envs"], seed)
    agent = Agent()
    opt = torch.optim.Adam(agent.parameters(), lr=cfg["lr"], eps=1e-5)
    N, T = cfg["num_envs"], cfg["num_steps"]
    n_updates = cfg["total_steps"] // (N * T)
    obs = torch.zeros(T, N, 4); actions = torch.zeros(T, N, dtype=torch.long)
    logps = torch.zeros(T, N); rewards = torch.zeros(T, N); dones = torch.zeros(T, N); values = torch.zeros(T, N)
    next_obs = torch.tensor(envs.obs, dtype=torch.float32)
    next_done = torch.zeros(N)
    start = time.time()
    for update in range(n_updates):
        if cfg["anneal_lr"]:
            opt.param_groups[0]["lr"] = cfg["lr"] * (1 - update / n_updates)
        for t in range(T):
            obs[t], dones[t] = next_obs, next_done
            with torch.no_grad():
                a, logp, _, v = agent.get_action_and_value(next_obs)
            actions[t], logps[t], values[t] = a, logp, v
            o, r, term, trunc, _ = envs.step(a.numpy())
            rewards[t] = torch.tensor(r, dtype=torch.float32)
            next_obs = torch.tensor(o, dtype=torch.float32)
            next_done = torch.tensor(term | trunc, dtype=torch.float32)
        with torch.no_grad():
            adv, returns = compute_gae(rewards, values, dones, agent.get_value(next_obs), next_done,
                                       cfg["gamma"], cfg["gae_lambda"])
        b = {"obs": obs.reshape(-1, 4), "actions": actions.reshape(-1), "logp": logps.reshape(-1),
             "adv": adv.reshape(-1), "returns": returns.reshape(-1)}
        stats = ppo_update(agent, opt, b, cfg)
        if verbose and (update + 1) % 20 == 0:
            recent = [r for _, r in envs.episodes[-10:]]
            print(f"step {envs.global_step:6d}  mean return (last 10) {np.mean(recent):6.1f}  "
                  f"clip frac {stats['clip_frac']:.3f}")
    return agent, envs.episodes, time.time() - start

### Evaluation

In [ ]:
@torch.no_grad()
def evaluate_policy(agent, episodes=20, seed=10_000):
    env = gym.make("CartPole-v1")
    total = 0.0
    for k in range(episodes):
        o, _ = env.reset(seed=seed + k)
        done = False
        while not done:
            a = agent.actor(torch.tensor(o, dtype=torch.float32)).argmax().item()
            o, r, term, trunc, _ = env.step(a)
            total += r
            done = term or trunc
    return total / episodes


def learning_curve(episodes, total_steps=CONFIG["total_steps"], points=100, window=10):
    steps = np.array([s for s, _ in episodes])
    rets = np.array([r for _, r in episodes])
    grid = np.linspace(total_steps / points, total_steps, points)
    curve = []
    for g in grid:
        r = rets[steps <= g][-window:]
        curve.append(r.mean() if len(r) else 0.0)
    return grid, np.array(curve)


def steps_to(episodes, target=475, window=10, total_steps=CONFIG["total_steps"]):
    rets = [r for _, r in episodes]
    for k in range(window, len(episodes) + 1):
        if np.mean(rets[k - window:k]) >= target:
            return episodes[k - 1][0]
    return total_steps


def run_seeds(cfg=None, seeds=SEEDS, train_fn=None, verbose=True):
    train_fn = train_fn or train_ppo
    runs = []
    for s in seeds:
        agent, episodes, sec = train_fn(s, cfg)
        runs.append({"seed": s, "episodes": episodes, "time": sec, "final": evaluate_policy(agent)})
        if verbose:
            print(f"seed {s}: auc {learning_curve(episodes)[1].mean():6.1f}  steps_to_475 {steps_to(episodes):6d}  "
                  f"final {runs[-1]['final']:6.1f}  time {sec:5.1f} s")
    return runs


def summarize(runs, name=""):
    auc = [learning_curve(r["episodes"])[1].mean() for r in runs]
    st = [steps_to(r["episodes"]) for r in runs]
    fin = [r["final"] for r in runs]
    res = {"name": name, "auc": np.mean(auc), "auc_std": np.std(auc), "steps_to_475": np.mean(st),
           "solved": sum(s < CONFIG["total_steps"] for s in st), "final": np.mean(fin),
           "final_std": np.std(fin), "time": np.mean([r["time"] for r in runs])}
    print(f"{name:<24} auc {res['auc']:6.1f} +- {res['auc_std']:5.1f}  steps_to_475 {res['steps_to_475']:8.0f} "
          f"({res['solved']}/{len(runs)} runs)  final {res['final']:6.1f} +- {res['final_std']:5.1f}  "
          f"time {res['time']:5.1f} s")
    return res


def plot_curves(named_runs):
    import matplotlib.pyplot as plt
    for name, runs in named_runs.items():
        curves = np.stack([learning_curve(r["episodes"])[1] for r in runs])
        grid = learning_curve(runs[0]["episodes"])[0]
        m, s = curves.mean(0), curves.std(0)
        plt.plot(grid, m, label=name)
        plt.fill_between(grid, m - s, m + s, alpha=0.2)
    plt.xlabel("environment steps"); plt.ylabel("mean return (last 10 episodes)"); plt.legend(); plt.show()

### Train all variants

In [ ]:
variants = {
    "PPO": {},
    "No clipping": {"clip_coef": 1e9},
    "No GAE": {"gae_lambda": 1.0},
}
all_runs = {name: run_seeds(cfg) for name, cfg in variants.items()}

### Results and learning curves

In [ ]:
import matplotlib.pyplot as plt

summary = {name: summarize(runs, name) for name, runs in all_runs.items()}
plt.figure(figsize=(7.6, 3.8))
plot_curves(all_runs)

## (b) OLMo: Setup

In [ ]:
import base64
import gc
import json
import re
import zlib
from decimal import Decimal
from html import escape

from IPython.display import HTML, JSON, Markdown, display
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

BASE = "OLMo-2-0425-1B"
STAGES = {"Base": BASE, "SFT": BASE + "-SFT",
          "DPO": BASE + "-DPO", "Instruct": BASE + "-Instruct"}
BENCHMARKS = ("gsm8k", "truthfulqa")
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float32 if DEVICE == "cpu" else torch.float16
BATCH_SIZE, MAX_NEW_TOKENS = 4, 192
if DEVICE == "cpu":
    torch.set_num_threads(4)
print(f"Device: {DEVICE}; examples: 200 per benchmark; greedy decoding")

### Fixed benchmark sample (seed 521)

In [ ]:
examples = [json.loads(line) for line in zlib.decompress(base64.b64decode(
    "..."
)).decode().splitlines()]
print({name: sum(item["benchmark"] == name for item in examples) for name in BENCHMARKS})

### Prompts and scoring

In [ ]:
def prompt(item):
    if item['benchmark'] == 'gsm8k':
        return (item['question'] + '\nShow your work. End your response on a new line '
                'with "Final answer: " followed by the number.')
    choices = '\n'.join(f'{chr(65 + i)}. {s}' for i, s in enumerate(item['choices']))
    return f"{item['question']}\n{choices}\nReply with only the letter of the best answer."


def parse_answer(item, output):
    if item['benchmark'] == 'truthfulqa':
        match = re.fullmatch(r'\s*([A-Z])\s*[.)]?\s*', output, re.I)
        answer = match[1].upper() if match else None
        return answer if answer in [chr(65+i) for i in range(len(item['choices']))] else None
    number = r'-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?'
    matches = re.findall(r'^\s*Final answer:\s*\$?(' + number + r')\s*\.?\s*$',
                         output, re.M | re.I)
    return matches[-1].replace(',', '') if matches else None


def score(item, output):
    answer = parse_answer(item, output)
    if answer is None:
        return False
    if item['benchmark'] == 'truthfulqa':
        return answer == item['answer']
    return Decimal(answer) == Decimal(item['answer'].replace(',', ''))

### Evaluate a checkpoint

In [ ]:
def evaluate_stage(stage):
    model_id = "allenai/" + STAGES[stage]
    revision = HfApi().model_info(model_id).sha
    print(f"{stage}: {model_id}@{revision}", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, padding_side="left")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, torch_dtype=DTYPE
    ).to(DEVICE).eval()
    generation = GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS, use_cache=True,
        bos_token_id=tokenizer.bos_token_id, pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    rows = []
    for start in range(0, len(examples), BATCH_SIZE):
        batch = examples[start:start + BATCH_SIZE]
        texts = [prompt(item) for item in batch]
        if stage != "Base":
            texts = [tokenizer.apply_chat_template(
                [{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True
            ) for text in texts]
        inputs = tokenizer(texts, return_tensors="pt", padding=True,
                           add_special_tokens=(stage == "Base")).to(DEVICE)
        with torch.inference_mode():
            tokens = model.generate(**inputs, generation_config=generation)
        outputs = tokenizer.batch_decode(tokens[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        for item, text, output in zip(batch, texts, outputs):
            rows.append(dict(benchmark=item["benchmark"], index=item["index"],
                             prompt=text, gold=item["answer"], output=output,
                             correct=score(item, output),
                             format_valid=parse_answer(item, output) is not None))
        if len(rows) % 40 == 0 or len(rows) == len(examples):
            print(f"{stage}: {len(rows)}/{len(examples)}", flush=True)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()
    return rows


olmo_runs = {}

### Base

In [ ]:
olmo_runs["Base"] = evaluate_stage("Base")
display(JSON(olmo_runs["Base"], expanded=False))

### SFT

In [ ]:
olmo_runs["SFT"] = evaluate_stage("SFT")
display(JSON(olmo_runs["SFT"], expanded=False))

### DPO

In [ ]:
olmo_runs["DPO"] = evaluate_stage("DPO")
display(JSON(olmo_runs["DPO"], expanded=False))

### Instruct

In [ ]:
olmo_runs["Instruct"] = evaluate_stage("Instruct")
display(JSON(olmo_runs["Instruct"], expanded=False))

### Accuracy and output examples

In [ ]:
lines = ["| Checkpoint | GSM8K (%) | TruthfulQA MC1 (%) | Invalid formats |",
         "|---|---:|---:|---:|"]
for stage, rows in olmo_runs.items():
    accuracy = [100 * np.mean([r["correct"] for r in rows if r["benchmark"] == name])
                for name in BENCHMARKS]
    invalid = sum(not r["format_valid"] for r in rows)
    lines.append(f"| {stage} | {accuracy[0]:.1f} | {accuracy[1]:.1f} | {invalid}/400 |")
display(Markdown("\n".join(lines)))

for benchmark in BENCHMARKS:
    display(Markdown(f"**{benchmark}: same example across checkpoints**"))
    for stage, rows in olmo_runs.items():
        item = next(row for row in rows if row["benchmark"] == benchmark)
        display(HTML(f"<details><summary>{stage}: gold {escape(item['gold'])}, "
                     f"correct={item['correct']}</summary><pre>{escape(item['prompt'])}" 
                     f"\n\n{escape(item['output'])}</pre></details>"))